In [5]:
%pip install pandas

import pandas as pd

df = pd.read_csv("gym.csv")

import os

# Explicitly set the working directory to your project root
project_dir = r"C:\Users\calum\gym-analytics"
os.chdir(project_dir)

print(f"Current working directory set to: {os.getcwd()}")

Note: you may need to restart the kernel to use updated packages.
Current working directory set to: C:\Users\calum\gym-analytics



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
cols_dropped = ["Kcal", "Distance", "Duration", "Bodyweight"]

df = df.drop(columns = cols_dropped)

df = df[df["Reps"].notnull()]

df["Name"] = df["Name"].str.strip()
sorted(df["Name"].dropna().unique())

df = df[df["Workout End"].notnull()]

df=df.sort_values("Workout Start").reset_index(drop=True)
df["workout_id"] = df.groupby(["Workout Start", "Workout End"]).ngroup()
df["workout_id"] = df["workout_id"].astype(int)
df["workout_id"] = df["workout_id"] + 1
print(df["workout_id"].nunique(), "unique workouts")

df = df.reset_index(drop=True)
df["set_id"] = df.index + 1

df["exercise_id"] = pd.factorize(
    df["workout_id"].astype(str) + "_" + df["Exercise"]
)[0] + 1

df = df.rename(columns={
    "Workout Start": "workout_start",
    "Workout End": "workout_end",
    "Exercise": "exercise",
    "Weight": "weight",
    "Reps": "reps",
    "Notes": "notes",
    "Category": "category",
    "Name": "workout_name"
})

df["workout_start"] = pd.to_datetime(df["workout_start"])
df["workout_end"] = pd.to_datetime(df["workout_end"])

df.to_csv("staged_workouts.csv", index=False)

797 unique workouts


In [7]:
import dlt

df2 = pd.read_csv("staged_workouts.csv")

pipeline = dlt.pipeline(
    pipeline_name = "gym",
    destination = "bigquery",
    dataset_name = "gym_raw",
    pipelines_dir= r"C:\Users\calum\gym-analytics\.dlt"
)

load_info = pipeline.run(df2.to_dict(orient="records"), table_name = "workouts")
print(load_info)

C:\Users\calum\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\auth\_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
C:\Users\calum\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\client.py:613: UserWarning: Cannot create BigQuery Storage client, the dependency google-cloud-bigquery-storage is not installed.
  warnings.warn(
C:\Users\calum\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\auth\_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the fo

Pipeline gym load step completed in 14.17 seconds
1 load package(s) were loaded to destination bigquery and into dataset gym_raw
The bigquery destination used None@None location to store data
Load package 1782076963.0911877 is LOADED and contains no failed jobs
